In [1]:
#| default_exp data

%load_ext autoreload
%autoreload 2
%run 99_util_imports.ipynb

In [11]:
import wandb
from torch.optim.lr_scheduler import StepLR


In [2]:
project="CIGNN_online_baseline"
max_epochs = 100 
batch_size = 32
patience = 3
n_runs_per_model = 3
learning_rate = 1e-6
min_increase = 1e-5
dev = "cuda"
env = "prod"

In [13]:
from CIGNN_experimental.unipen_renderer.parser import Parser
from CIGNN_experimental.unipen_renderer.strokes_to_image_converter import Strokes2ImageConverter
data = np.load("../CIGNN_experimental/unipen_renderer/unipen_strokes.npy", allow_pickle=True).item()

# re-index the data 
keys = list(data.keys())
keys.sort()
data = {idx: data[cls] for idx, cls in enumerate(keys)}

# create X, y vectors for splitting
X, y = [], []
for cls, splines in data.items():
    for spline in splines:
        X.append(spline)
        y.append(cls)

In [14]:
# Split into train, validation and test
# Got to call it twice, since sklearn does not have a triple-split builtin
# Use random_state=42 for reproducability
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42) 

In [15]:
# Wrapper for torch to use this in a dataloader
class UnipenDataset(Dataset):
    def __init__(self, population, image_size=(64, 64)):
        """
        """
        from torchvision.transforms.functional import pil_to_tensor
        self.data = []
        s2i = Strokes2ImageConverter()
        for sample, label in population:
            img = s2i.strokes_to_image(strokes=sample, image_size=image_size, smoothing_factor=0.0, stroke_width=3).convert("RGB")
            # Finally, the image needs to be converted to tensors
            tensor = pil_to_tensor(img)
            self.data.append((tensor, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, label = self.data[idx]
        return img, label


In [16]:
# Set up datasets and dataloaders
# The image size of 244, 244 is required by the transformer model later on
n_classes = np.unique(y_train).size
train_ds = UnipenDataset(zip(X_train, y_train), image_size=(224, 224))
val_ds = UnipenDataset(zip(X_val, y_val), image_size=(224, 224))
test_ds = UnipenDataset(zip(X_test, y_test), image_size=(224, 224))
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [17]:
device = torch.device("cuda")

In [ ]:
# Train model: VGG 16 BN [CNN-based]

# Prepare training and init model
model = torchvision.models.vgg16_bn(num_classes=n_classes)
model_name = "vgg16bn"
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Save the initial model to re-load the exact parameters 
initial_checkpoint_path = f"31_CIGNN_01_{model_name}_init.pth"
optimal_checkpoint_path = f"31_CIGNN_01_{model_name}_best.pth" 
torch.save({
    'model_state_dict': model.state_dict(),
}, initial_checkpoint_path) 

for run_i in range(n_runs_per_model):

    run = wandb.init(
        project="CIGNN Unipen Baselines",
        name=f"{model_name} run {run_i}"
    )

    # Reset state to assert each run starts under the same circumstances
    del model 
    del optimizer
    torch.cuda.empty_cache()

    model = torchvision.models.vgg16_bn(num_classes=n_classes)
    model.to(device)
    checkpoint = torch.load(initial_checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)   
    scheduler = StepLR(optimizer, step_size=10, gamma=0.1)


    best_val_loss = float('inf')  # Initialize the best validation loss
    counter = 0   # Counter for consecutive epochs without improvement (required for patience)
    
    for epoch in range(max_epochs):
        model.train()
    
        # Training phase
        for batch_idx, (images, labels) in enumerate(train_dl, start=1):
            images, labels = images.to(device).to(torch.float), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
   
            if batch_idx % 10 == 0:
                wandb.log({f"{model_name} train_loss": loss.item()})
                    
    
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_dl:
                images, labels = images.to(device).to(torch.float), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
    
        avg_val_loss = val_loss / len(val_dl)
        wandb.log({f"{model_name} val_loss": avg_val_loss})
        scheduler.step()
    
        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), optimal_checkpoint_path)
            counter = 0  # Reset counter if performance improves
        else:
            counter += 1
            if counter >= patience:
                break
                
    # Testing loop 
    # calculate accuracy and f1 score for the best saved model checkpoint
    model.load_state_dict(torch.load(optimal_checkpoint_path))  # Load the best model checkpoint
    model.eval()
    correct = 0
    total = 0
    all_labels = []
    all_predictions = []
    
    with torch.no_grad():
        for images, labels in test_dl:
            images, labels = images.to(device).to(torch.float), labels.to(device)
            outputs = model(images)  # Forward pass
            predicted = outputs.argmax(dim=1)  # Get predicted class
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
            # Collect all labels and predictions for metrics calculation
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
    
    # Compute overall accuracy
    accuracy = 100 * correct / total
    
    # Compute F1 scores per class
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    
    f1_micro = f1_score(all_labels, all_predictions, average="micro")
    wandb.log({f"{model_name} f1_micro": f1_micro})
    # Log overall F1 score (macro-averaged)
    f1_macro = f1_score(all_labels, all_predictions, average="macro")
    wandb.log({f"{model_name} f1_macro": f1_macro})
    
    # Compute and log the confusion matrix
    conf_matrix = confusion_matrix(all_labels, all_predictions)   
    wandb.log({f"{model_name} confusion_matrix": wandb.Table(data=conf_matrix.tolist(), columns=[f"Class_{i}" for i in range(len(conf_matrix))])})

    # Finally, save the model to wandb and terminate the run
    wandb.log_model(optimal_checkpoint_path, f"{model_name}_r{run_i}")
    run.finish()

In [ ]:
# Train model: GoogLeNet [CNN-based]

# Prepare training and init model
model = torchvision.models.googlenet(num_classes=n_classes, aux_logits=False)
model_name = "GoogLeNet"
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Save the initial model to re-load the exact parameters 
initial_checkpoint_path = f"31_CIGNN_01_{model_name}_init.pth"
optimal_checkpoint_path = f"31_CIGNN_01_{model_name}_best.pth" 
torch.save({
    'model_state_dict': model.state_dict(),
}, initial_checkpoint_path) 

for run_i in range(n_runs_per_model):

    run = wandb.init(
        project="CIGNN Unipen Baselines",
        name=f"{model_name} run {run_i}"
    )

    # Reset state to assert each run starts under the same circumstances
    del model 
    del optimizer
    torch.cuda.empty_cache()

    model = torchvision.models.googlenet(num_classes=n_classes, aux_logits=False)
    model.to(device)
    checkpoint = torch.load(initial_checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)   
    scheduler = StepLR(optimizer, step_size=10, gamma=0.1)


    best_val_loss = float('inf')  # Initialize the best validation loss
    counter = 0   # Counter for consecutive epochs without improvement (required for patience)
    
    for epoch in range(max_epochs):
        model.train()
    
        # Training phase
        for batch_idx, (images, labels) in enumerate(train_dl, start=1):
            images, labels = images.to(device).to(torch.float), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
   
            if batch_idx % 10 == 0:
                wandb.log({f"{model_name} train_loss": loss.item()})
                    
    
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_dl:
                images, labels = images.to(device).to(torch.float), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
    
        avg_val_loss = val_loss / len(val_dl)
        wandb.log({f"{model_name} val_loss": avg_val_loss})
        scheduler.step()
    
        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), optimal_checkpoint_path)
            counter = 0  # Reset counter if performance improves
        else:
            counter += 1
            if counter >= patience:
                break
                
    # Testing loop 
    # calculate accuracy and f1 score for the best saved model checkpoint
    model.load_state_dict(torch.load(optimal_checkpoint_path))  # Load the best model checkpoint
    model.eval()
    correct = 0
    total = 0
    all_labels = []
    all_predictions = []
    
    with torch.no_grad():
        for images, labels in test_dl:
            images, labels = images.to(device).to(torch.float), labels.to(device)
            outputs = model(images)  # Forward pass
            predicted = outputs.argmax(dim=1)  # Get predicted class
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
            # Collect all labels and predictions for metrics calculation
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
    
    # Compute overall accuracy
    accuracy = 100 * correct / total
    
    # Compute F1 scores per class
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    
    f1_micro = f1_score(all_labels, all_predictions, average="micro")
    wandb.log({f"{model_name} f1_micro": f1_micro})
    # Log overall F1 score (macro-averaged)
    f1_macro = f1_score(all_labels, all_predictions, average="macro")
    wandb.log({f"{model_name} f1_macro": f1_macro})
    
    # Compute and log the confusion matrix
    conf_matrix = confusion_matrix(all_labels, all_predictions)   
    wandb.log({f"{model_name} confusion_matrix": wandb.Table(data=conf_matrix.tolist(), columns=[f"Class_{i}" for i in range(len(conf_matrix))])})

    # Finally, save the model to wandb and terminate the run
    wandb.log_model(optimal_checkpoint_path, f"{model_name}_r{run_i}")
    run.finish()

In [ ]:
# Train model: ViT-B/16  [Transformer-based]

# Prepare training and init model
model = torchvision.models.vit_b_16(num_classes=n_classes)
model_name = "ViT_b16"
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Save the initial model to re-load the exact parameters 
initial_checkpoint_path = f"31_CIGNN_01_{model_name}_init.pth"
optimal_checkpoint_path = f"31_CIGNN_01_{model_name}_best.pth" 
torch.save({
    'model_state_dict': model.state_dict(),
}, initial_checkpoint_path) 

for run_i in range(n_runs_per_model):

    run = wandb.init(
        project="CIGNN Unipen Baselines",
        name=f"{model_name} run {run_i}"
    )

    # Reset state to assert each run starts under the same circumstances
    del model 
    del optimizer
    torch.cuda.empty_cache()

    model = torchvision.models.vit_b_16(num_classes=n_classes)
    model.to(device)
    checkpoint = torch.load(initial_checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = StepLR(optimizer, step_size=10, gamma=0.1)


    best_val_loss = float('inf')  # Initialize the best validation loss
    counter = 0   # Counter for consecutive epochs without improvement (required for patience)
    
    for epoch in range(max_epochs):
        model.train()
    
        # Training phase
        for batch_idx, (images, labels) in enumerate(train_dl, start=1):
            images, labels = images.to(device).to(torch.float), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
   
            if batch_idx % 10 == 0:
                wandb.log({f"{model_name} train_loss": loss.item()})
                    
    
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_dl:
                images, labels = images.to(device).to(torch.float), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
    
        avg_val_loss = val_loss / len(val_dl)
        wandb.log({f"{model_name} val_loss": avg_val_loss})
        scheduler.step()
    
        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), optimal_checkpoint_path)
            counter = 0  # Reset counter if performance improves
        else:
            counter += 1
            if counter >= patience:
                break
                
    # Testing loop 
    # calculate accuracy and f1 score for the best saved model checkpoint
    model.load_state_dict(torch.load(optimal_checkpoint_path))  # Load the best model checkpoint
    model.eval()
    correct = 0
    total = 0
    all_labels = []
    all_predictions = []
    
    with torch.no_grad():
        for images, labels in test_dl:
            images, labels = images.to(device).to(torch.float), labels.to(device)
            outputs = model(images)  # Forward pass
            predicted = outputs.argmax(dim=1)  # Get predicted class
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
            # Collect all labels and predictions for metrics calculation
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
    
    # Compute overall accuracy
    accuracy = 100 * correct / total
    
    # Compute F1 scores per class
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    
    f1_micro = f1_score(all_labels, all_predictions, average="micro")
    wandb.log({f"{model_name} f1_micro": f1_micro})
    # Log overall F1 score (macro-averaged)
    f1_macro = f1_score(all_labels, all_predictions, average="macro")
    wandb.log({f"{model_name} f1_macro": f1_macro})
    
    # Compute and log the confusion matrix
    conf_matrix = confusion_matrix(all_labels, all_predictions)   
    wandb.log({f"{model_name} confusion_matrix": wandb.Table(data=conf_matrix.tolist(), columns=[f"Class_{i}" for i in range(len(conf_matrix))])})

    # Finally, save the model to wandb and terminate the run
    wandb.log_model(optimal_checkpoint_path, f"{model_name}_r{run_i}")
    run.finish()